# IOAI — 2025 Selection 2 Eye For Feature Engineering (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-selection-2-eye-for-feature-engineering'
for f in ['train.csv','test.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# An eye for feature engineering — MAIO 2025 (Selection 2)

기이한 2D 데이터(`feature1`, `feature2`)를 로지스틱 회귀로 분류한다. 베이스라인은 두 feature 만으로
학습해 **F1 = 0** 이다. 규칙은 원본과 같다:

- **`create_new_feature(X)` 셀만** 고쳐 새 feature(`feature3`)를 만든다. 모델 로직(로지스틱 회귀)은 그대로 둔다.
- 목표: `feature3` 을 잘 설계해 **F1 을 최대한 높인다**.

여기서는 원본 148행을 train/test 로 나눠(테스트 라벨 숨김) held-out **F1** 로 채점한다.
`data/train.csv`(라벨 포함) 로 학습하고 `data/test.csv` 를 예측해 `data/submission.csv`(`id,class`) 를 만든다.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

train = pd.read_csv("data/train.csv")   # id, feature1, feature2, class
test  = pd.read_csv("data/test.csv")    # id, feature1, feature2  (class 숨김)

In [ ]:
# ★ 이 셀만 고치세요 — feature1, feature2 로부터 새 feature 를 만든다.
def create_new_feature(X):
    return X["feature1"]        # (베이스라인: 의미 없는 복제 → F1 = 0)

In [ ]:
FEATURES = ["feature1", "feature2", "feature3"]

def fit_predict():
    Xtr = train[["feature1", "feature2"]].copy()
    Xte = test[["feature1", "feature2"]].copy()
    Xtr["feature3"] = create_new_feature(Xtr)
    Xte["feature3"] = create_new_feature(Xte)
    logreg = LogisticRegression(max_iter=2000)
    logreg.fit(Xtr[FEATURES], train["class"])
    return logreg.predict(Xtr[FEATURES]), logreg.predict(Xte[FEATURES])

train_pred, test_pred = fit_predict()
print("train F1:", round(f1_score(train["class"], train_pred, zero_division=0), 4))

In [ ]:
sub = pd.DataFrame({"id": test["id"], "class": test_pred.astype(int)})
sub.to_csv("submission.csv", index=False)
print("saved data/submission.csv", sub.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)